# CyBench Agent Architecture Analysis (Domain-Specific)

## Purpose

This notebook contains **CyBench domain-specific** experiments comparing different **agent architectures** (React, GH Copilot, Claude Code, etc.).

It focuses on analyses that are unique to CyBench and complement the domain-agnostic agent architecture analysis notebook (`agent_architecture_analysis.ipynb`).

### Experiments in this notebook

| # | Experiment | Why domain-specific |
|---|-----------|---------------------|
| 1 | Submission vs. Checkpoint Gap Heatmap | Per-challenge × per-agent gap analysis unique to CyBench's 6-checkpoint structure |


## Prerequisites

1. Run evaluations for each agent architecture to generate `.eval` files:
   ```bash
   uv run inspect eval domains/cybench --model anthropic/claude-sonnet-4-6 -T agent=react -T build=true --display plain
   uv run inspect eval domains/cybench --model anthropic/claude-sonnet-4-6 -T agent=copilot --display plain
   ```
2. Place eval files in `eval_samples/cybench/` or `latest_experiments/cybench/`
3. Run the domain-agnostic [`agent_architecture_analysis.ipynb`](agent_architecture_analysis.ipynb) first for overall comparisons across all experiments (1–12)

<a id="configuration"></a>
## Configuration

Edit the cell below to configure agents, models, and eval file paths for CyBench.

### How to add agents or models

1. Add entries to `AGENT_EVAL_LOGS` under the appropriate agent architecture
2. Ensure the corresponding `.eval` files exist in `eval_samples/cybench/` or `latest_experiments/cybench/`
3. The flat `EVAL_LOGS` dict is auto-generated as `"Model (Agent)"` → filename

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Agent Architecture Analysis (CyBench-Specific)
# ══════════════════════════════════════════════════════════════════════════════
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from inspect_ai.log import read_eval_log

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from saber_analysis.data_loader import load_eval_logs, load_trajectory_data, ensure_eval_files
from saber_analysis.cost import extract_cost_rows
from saber_analysis.plots import setup_plotting, classify_score_type

setup_plotting()

# ── Domain ──
DOMAIN_NAME = 'CyBench'
DOMAIN_SLUG = 'cybench'
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
LOG_DIR = str(REPO_ROOT / 'eval_samples' / DOMAIN_SLUG)
ARTIFACTS_DIR = NOTEBOOK_DIR / 'artifacts' / 'cybench_agent'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# AGENT REGISTRY — {Display Name: filename.eval} organized by agent
# ══════════════════════════════════════════════════════════════════════════════
AGENT_EVAL_LOGS = {
    'React': {
        'Claude Haiku 4.5':  'haiku_4.5_react.eval',
        'Claude Sonnet 4.6': 'sonnet_4.6_react.eval',
        'Claude Opus 4.6':   'opus_4.6_react.eval',
        'GPT-5.4':           'gpt_5.4_react.eval',
    },
    'GH Copilot': {
        'Claude Haiku 4.5':  'haiku_4.5_copilot.eval',
        'Claude Sonnet 4.6': 'sonnet_4.6_copilot.eval',
        'Claude Opus 4.6':   'opus_4.6_copilot.eval',
        'GPT-5.4':           'gpt_5.4_copilot.eval',
        'GPT-5.4-mini':      'gpt_5.4_mini_copilot.eval',
    },
    # 'Claude Code': {
    #     'Claude Sonnet 4.6': 'sonnet_4.6_claude_code.eval',
    # },
}

AGENT_COLORS = {
    'React':      '#2CA02C',
    'GH Copilot': '#E74C3C',
    'Claude Code':'#3498DB',
}

MODEL_COLORS = {
    'Claude Haiku 4.5':  '#17BECF',
    'Claude Sonnet 4.6': '#F39C12',
    'Claude Opus 4.6':   '#E74C3C',
    'GPT-5.4':           '#2CA02C',
    'GPT-5.4-mini':      '#7B4FBF',
}

# Flatten for data loading
EVAL_LOGS = {}
DISPLAY_ORDER = []
COLORS = {}
for agent, models in AGENT_EVAL_LOGS.items():
    for model, filename in models.items():
        display = f'{model} ({agent})'
        EVAL_LOGS[display] = filename
        DISPLAY_ORDER.append(display)
        COLORS[display] = MODEL_COLORS.get(model, '#888888')

MODEL_ORDER = DISPLAY_ORDER

# Grouping for CyBench: group by challenge name
GROUP_FN = lambda sid: '_'.join(sid.split('_')[:-2])
GROUP_LABEL = 'Challenge'

# Detect N_SAMPLES from first available eval
_found_first = False
for _fn in EVAL_LOGS.values():
    _p = os.path.join(LOG_DIR, _fn)
    if not os.path.exists(_p):
        _fallback = str(REPO_ROOT / 'latest_experiments' / DOMAIN_SLUG / _fn)
        if os.path.exists(_fallback):
            _p = _fallback
        else:
            continue
    _hdr = read_eval_log(_p, header_only=True)
    N_SAMPLES = len(_hdr.samples) if hasattr(_hdr, 'samples') and _hdr.samples else 0
    if N_SAMPLES == 0 and hasattr(_hdr, 'results') and _hdr.results:
        N_SAMPLES = _hdr.results.completed_samples or 0
    TOOL_CALL_LIMIT = _hdr.eval.config.tool_call_limit or 25
    _found_first = True
    break

if not _found_first:
    N_SAMPLES = 0
    TOOL_CALL_LIMIT = 25
    print('⚠ No eval files found — set N_SAMPLES manually')

# Download/locate eval files
ensure_eval_files(
    EVAL_LOGS, LOG_DIR, DOMAIN_SLUG,
    fallback_dirs=[str(REPO_ROOT / 'latest_experiments' / DOMAIN_SLUG)],
)

print(f'✓ Domain: {DOMAIN_NAME}')
print(f'  Agent configs: {len(EVAL_LOGS)} ({", ".join(DISPLAY_ORDER)})')
print(f'  Log dir: {LOG_DIR}')


## Data Loading

### What this does

Parses all `.eval` ZIP archives from the agent × model matrix and extracts:
- **`samples_df`**: per-sample `saber_overall` scores with config and challenge group columns
- **`subtasks_df`**: per-subtask scores (submission, 6 checkpoints, aggregate)
- **`overall_df`**: per-config mean and standard error

### Why

These DataFrames support the CyBench-specific gap heatmap below. The `model` column encodes both model identity and agent architecture (e.g., `Claude Sonnet 4.6 (React)`).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════
samples_df, subtasks_df, overall_df, model_overall = load_eval_logs(
    EVAL_LOGS, LOG_DIR, MODEL_ORDER, group_fn=GROUP_FN
)

print(f'\n✓ Loaded {len(samples_df)} sample scores across {samples_df["model"].nunique()} agent configs')
print(f'✓ Loaded {len(subtasks_df)} sub-task scores')
if GROUP_FN:
    print(f'✓ Groups: {sorted(samples_df["group"].unique())}')
print()
print('═══ Overall Scores per Agent Config ═══')
for m in MODEL_ORDER:
    if m in model_overall:
        r = overall_df.loc[m]
        print(f'  {m:35s}  mean={r["mean"]:.4f}  stderr={r["stderr"]:.4f}')

# Classify subtasks
subtasks_df['category'] = subtasks_df['score_type'].apply(classify_score_type)


---

## 1. Submission vs. Checkpoint Gap Analysis (CyBench-Specific)

### What this measures

This heatmap shows the gap between submission and checkpoint scores **per challenge per agent configuration**.

- **Red (positive):** Submission > Checkpoints — agent captured the flag without thorough investigation steps
- **Green (negative):** Checkpoints > Submission — agent progressed through steps but failed the final exploit
- **Yellow (≈0):** Aligned — investigation quality matches exploit success

CyBench has 6 checkpoints per challenge (contextualizing, understanding templating, identifying attack vectors, exploring templates, locating the flag, and crafting the exploit), making this gap particularly informative.

### Agent Architecture Dimension

By comparing across agent architectures, we can see whether certain agent designs are more prone to "guessing" (positive gaps) vs. thorough investigation (negative gaps).

### Key insights (CyBench)

- No CyBench agent architecture eval files are currently available in eval_samples/
- This analysis will be populated when React and GH Copilot agent evals are run on CyBench challenges
- Check latest_experiments/cybench/ for any available agent runs that could be added


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1 (CYBENCH-SPECIFIC): Submission vs checkpoint gap heatmap — per agent
# ══════════════════════════════════════════════════════════════════════════════
sub_means = subtasks_df[subtasks_df['category'] == 'Submission'].groupby(
    ['group', 'model'])['score'].mean().unstack('model').reindex(columns=MODEL_ORDER)
cp_means = subtasks_df[subtasks_df['category'] == 'Checkpoints'].groupby(
    ['group', 'model'])['score'].mean().unstack('model').reindex(columns=MODEL_ORDER)
gap = sub_means - cp_means

# Drop columns/rows that are entirely NaN
gap = gap.dropna(axis=1, how='all').dropna(axis=0, how='all')
active_models = [m for m in MODEL_ORDER if m in gap.columns]

fig, ax = plt.subplots(figsize=(max(14, len(gap.index) * 1.2), max(7, len(active_models) * 0.8)))
im = ax.imshow(gap[active_models].T.values, cmap='RdYlGn_r', aspect='auto', vmin=-0.3, vmax=0.3)
ax.set_xticks(range(len(gap.index)))
ax.set_xticklabels(gap.index, fontsize=8, rotation=45, ha='right')
ax.set_yticks(range(len(active_models)))
ax.set_yticklabels(active_models, fontsize=9)

for i, model in enumerate(active_models):
    for j, challenge in enumerate(gap.index):
        val = gap.loc[challenge, model]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.2f}', ha='center', va='center', fontsize=7,
                    fontweight='bold', color='white' if abs(val) > 0.15 else 'black')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Submission − Checkpoint Gap', fontsize=10)
ax.set_title('Submission vs Checkpoint Gap per Challenge (by Agent Architecture)\n'
             '(positive = flag captured without thorough investigation)', fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'submission_checkpoint_gap_by_agent.png', bbox_inches='tight')
plt.show()

print('═══ Average Submission–Checkpoint Gap per Agent Config ═══')
for m in active_models:
    print(f'  {m:35s}  avg gap = {gap[m].mean():+.4f}')

# ── Agent-level summary ──
print()
print('═══ Average Gap by Agent Architecture ═══')
for agent, models in AGENT_EVAL_LOGS.items():
    agent_cols = [f'{m} ({agent})' for m in models if f'{m} ({agent})' in gap.columns]
    if agent_cols:
        agent_gap = gap[agent_cols].mean().mean()
        print(f'  {agent:15s}  avg gap = {agent_gap:+.4f}')


---

## Next Steps

### Companion notebooks

- Run the **domain-agnostic** [`agent_architecture_analysis.ipynb`](agent_architecture_analysis.ipynb) for overall score comparisons, cost analysis, tool usage, and trajectory analysis across agent architectures (Experiments 1–12).
- See [`cybench_analysis.ipynb`](cybench_analysis.ipynb) for model-comparison analysis in the CyBench domain (without the agent architecture dimension).

### Extending this analysis

1. After more agent evaluations complete, re-run this notebook to see gap patterns across architectures
2. Compare gap distributions to identify which agent designs are more "thorough" vs. "guess-prone" in CyBench challenges
3. With more CyBench challenges added, the heatmap will expand to reveal per-challenge × per-agent patterns
4. Cross-reference with the safety refusal analysis in [`model_safety_filters_analysis.ipynb`](model_safety_filters_analysis.ipynb) to understand if any agent architectures trigger refusals on CyBench tasks